# Basic Neural Network

In [630]:
import idx2numpy # type: ignore
import numpy as np
import numpy.linalg as la

import pandas as pd

import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pio.templates.default = "plotly_dark" # type: ignore

In [631]:
# (60000, 28, 28) images, (60000,) labels
images_2d = idx2numpy.convert_from_file("../data/train-images.idx3-ubyte")  # type: ignore
images = images_2d.reshape(images_2d.shape[0], -1)  # type: ignore
images_scaled = images / 255
labels = idx2numpy.convert_from_file("../data/train-labels.idx1-ubyte")  # type: ignore

test_images_2d = idx2numpy.convert_from_file("../data/t10k-images.idx3-ubyte")  # type: ignore
test_images = test_images_2d.reshape(test_images_2d.shape[0], -1)  # type: ignore
test_images_scaled = test_images / 255
test_labels = idx2numpy.convert_from_file("../data/t10k-labels.idx1-ubyte")  # type: ignore

In [3]:
images_shown = 5

# 1. Create a subplot grid with 1 row and 'images_shown' columns
fig = make_subplots(
    rows=1, 
    cols=images_shown,
    subplot_titles=[f"Label: {l}" for l in labels[:images_shown]]
)

for i in range(images_shown):
    img_trace = go.Heatmap(
            z=images_2d[i], 
            colorscale="gray", 
            showscale=False
        )
    fig.add_trace(img_trace, row=1, col=i + 1)


fig.update_yaxes(autorange="reversed", showticklabels=False)
fig.update_xaxes(showticklabels=False)
fig.update_layout(
    coloraxis_showscale=False,
    height=300,
    width=images_shown * 250,
    margin=dict(l=20, r=20, t=50, b=20),
)

fig.show()

## One Layer

In [4]:
output_layer_test = 16
weights_test = np.random.randn(output_layer_test, images.shape[1]) * .1
bias_test = np.random.randn(output_layer_test)

transform = weights_test @ images[0] + bias_test
output_test = 1 / (1 + np.exp(-transform))
output_test[0]

np.float64(3.041785458469592e-31)

# Multiple Layers

In [5]:
def calculate_layer(input: np.ndarray, weights: np.ndarray, bias: np.ndarray) -> np.ndarray:
    transform = weights @ input + bias
    output = 1 / (1 + np.exp(-transform))
    return output

In [31]:
middle_layer = 16
final_layer = 10

w_1 = np.random.randn(middle_layer, images.shape[1]) * 0.1
b_1 = np.random.randn(middle_layer) * 0.1
layer_1 = calculate_layer(
    images[0],
    w_1,
    b_1,
)
w_2 = np.random.randn(middle_layer, middle_layer) * 0.1
b_2 = np.random.randn(middle_layer) * 0.1
layer_2 = calculate_layer(
    layer_1,
    w_2,
    b_2,
)
w_3 = np.random.randn(final_layer, middle_layer) * 0.1
b_3 = np.random.randn(final_layer) * 0.1
final_layer = calculate_layer(
    layer_2,
    w_3,
    b_3,
)
final_layer

array([0.5428583 , 0.54445693, 0.324224  , 0.60360131, 0.45989844,
       0.51018834, 0.55233893, 0.48392543, 0.43863628, 0.53096397])

In [32]:
print(np.argmax(final_layer))
print(labels[0])

3
5


Gave wrong answer as weights are completely random

In [33]:
cost = np.mean(np.square(final_layer - np.eye(10)[labels[0]]))
cost

np.float64(0.2525592893706158)

## Backpropagation

We're trying to find the change of the cost based of the weights $ = \frac{\partial C}{\partial w}$. Or the gradient would say the movement of the weights that would create the biggest change in the cost.

Due to chain rule it turns into $\frac{\partial C}{\partial w} = \frac{\partial z}{\partial w}\frac{\partial a}{\partial z}\frac{\partial C}{\partial a}$ since $z_n = w_n a_{n-1} + b$ and $a_n = \sigma(z_n)$

$C = (a - y)^2$ where $\frac{\partial C}{\partial a} = 2(a - y)$

$a = \sigma(z)$ where $\frac{\partial a}{\partial z} = \sigma'(z)$ here $\sigma(z) = \frac{1}{1+e^{-x}}$ which is $\sigma'(z) = \sigma(z) * (1 - \sigma(z))$

$z = w * a + b$ where $\frac{\partial z}{\partial w} = a$

$\frac{\partial C}{\partial w} = a_{n-1} * \sigma'(z_n) * 2(a_n - y)$

One row manually

In [ ]:
d_cost = 2 * (final_layer - np.eye(10)[labels[0]])

z_3 = w_3 @ layer_2 + b_3
d_sigma3 = (1 / (1 + np.exp(-z_3))) * (1 - (1 / (1 + np.exp(-z_3))))

z_2 = w_2 @ layer_1 + b_2
d_sigma2 = (1 / (1 + np.exp(-z_2))) * (1 - (1 / (1 + np.exp(-z_2))))

z_1 = w_1 @ images[0] + b_1
d_sigma1 = (1 / (1 + np.exp(-z_1))) * (1 - (1 / (1 + np.exp(-z_1))))

Though it would be dc_dw = images[0] @ d_sigma1 @ d_sigma2 * d_sigma1

But the matrix and vectors don't line up

I kinda understand it a bit better now the final layer C is the cost

the cost here is (y - a)^2 or the difference between what it actually is and what it should be

we had the solution here be 5 and we got a different number so for a single neuron that is not 5 we would do (y - 0)^2 where y is value of that specific neuron

so based of that scalar cost value we know for that neuron all the weights attatched should changes based of that value and it works since a stronger weight is going to change stronger than a weaker weight

so dc/da is going to be vector with component of scalar and so is $\sigma'$ then we end up multipliying thoese so here in the last layer we have 10 elements each and for each one we need to multiply row wise and create another 10 element vector

then we next otuput should give us our change in weights. Basically this cost (and sigma) basically gives us how to change the w values. We multipply by the preivous layer and not straight to w which is what I was thinking but what if the previous layer didn't even activate the neuron so the weight was never used. so past how much the weight should change we should check how much that neuron was also used.

so we end up getting a matrix where each vector is the previous layer multiplied by the cost to get the change in weight matrix. So we probably need to do outer product to get a matrix.

In [58]:
print((d_cost))
print((d_cost).shape)

[ 1.0857166   1.08891386  0.64844799  1.20720262  0.91979688 -0.97962331
  1.10467786  0.96785086  0.87727255  1.06192794]
(10,)


In [68]:
print((d_sigma3))
print((d_sigma3).shape)

[0.24816317 0.24802358 0.2191028  0.23926677 0.24839186 0.2498962
 0.24726064 0.24974161 0.24623449 0.24904123]
(10,)


In [71]:
print((d_sigma3 * d_cost))
print((d_sigma3 * d_cost).shape)

[ 0.26943487  0.27007632  0.14207677  0.28884347  0.22847006 -0.24480414
  0.27314335  0.24171263  0.21601476  0.26446384]
(10,)


In [72]:
print((layer_2))
print((layer_2).shape)

[0.42748327 0.50387086 0.46863964 0.47899829 0.45918561 0.46780039
 0.44688684 0.4911542  0.68582863 0.48630404 0.49804496 0.47007209
 0.41815894 0.40302265 0.56764225 0.57426802]
(16,)


In [86]:
print(np.outer(layer_2, (d_sigma3 * d_cost))[0])
print((np.outer(layer_2, (d_sigma3 * d_cost))).shape)

[ 0.1151789   0.11545311  0.06073544  0.12347575  0.09766713 -0.10464968
  0.11676421  0.10332811  0.0923427   0.11305387]
(16, 10)


This works and finally makes sense but I also reallized that we have to take this new dc/dw or change in weight based of the cost and compute the previous layer.

So we need to actually find the dc/da for the next layer to use like we did at the start. So we are actually find dc/da till the last layers

$\frac{\partial C_0}{\partial w^{(L-1)}} = \frac{\partial z^{(L-1)}}{\partial w^{(L-1)}} \frac{\partial a^{(L-1)}}{\partial z^{(L-1)}} \frac{\partial z^{(L)}}{\partial a^{(L-1)}} \frac{\partial a^{(L)}}{\partial z^{(L)}} \frac{\partial C_0}{\partial a^{(L)}}$

here the dc/da becomes the weights also we need to tranpose the weight since it usually is 784 by 100 but we need to go to 100 by 784

In [80]:
print(np.outer(w_3, (d_sigma3 * d_cost).T)[0])
print((np.outer(w_3, (d_sigma3 * d_cost).T)).shape)

[ 0.02737818  0.02744336  0.0144369   0.02935035  0.02321561 -0.02487537
  0.02775501  0.02456123  0.02194999  0.02687306]
(160, 10)


In [99]:
print((w_3.T @ (d_sigma3 * d_cost))[0])
print((w_3.T @ (d_sigma3 * d_cost)).shape)

0.03375934973853483
(16,)


This vector we just keep going with to the intial weight change also fixed the outer since we were getting 728 by 16 but the actual weights are 16 by 728 since (16 * 728) * (728, 1) whcih is input gives us the second layer of 16 neurons.

In [101]:
a_3_change = w_3.T @ (d_sigma3 * d_cost)
a_2_change = w_2.T @ (d_sigma2 * a_3_change)
w_1_change = np.outer((d_sigma1 * a_2_change), images[0])
b_1_change = (d_sigma1 * a_2_change)
w_1_change.shape

(16, 784)

In [114]:
learning_rate = 1

w_1_new = w_1 - (learning_rate * w_1_change)
b_1_new = b_1 - (learning_rate * b_1_change)

w_2_new = w_2 - (learning_rate * np.outer((d_sigma2 * a_3_change), layer_1))
b_2_new = b_2 - (learning_rate * (d_sigma2 * a_3_change))

w_3_new = w_3 - (learning_rate * np.outer((d_sigma3 * d_cost), layer_2))
b_3_new = b_3 - (learning_rate * (d_sigma3 * d_cost))

In [115]:
layer_1_new = calculate_layer(
    images[0],
    w_1_new,
    b_1_new,
)
layer_2_new = calculate_layer(
    layer_1_new,
    w_2_new,
    b_2_new,
)
final_layer_new = calculate_layer(
    layer_2_new,
    w_3_new,
    b_3_new,
)
final_layer_new

array([0.23826977, 0.23907082, 0.1956452 , 0.26488227, 0.2130369 ,
       0.77567593, 0.24097805, 0.21728575, 0.21045939, 0.23336035])

In [116]:
print(np.argmax(final_layer_new))
print(labels[0])

5
5


It actually worked with a pretty high learning score butu should set it low in the future if we're doing multiple numbers

# Actual Code

In [3]:
def sigmoid(x: np.ndarray):
    return 1 / (1 + np.exp(-x))

def d_sigmoid(x: np.ndarray):
    return sigmoid(x) * (1 - sigmoid(x))

In [4]:
def neural_step(input_data: np.ndarray, label: int, weights: list[np.ndarray], biases: list[np.ndarray], learning_rate: float):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)
    z3 = W3 @ a2 + b3
    a3 = sigmoid(z3)

    d_c = 2 * (a3 - np.eye(10)[label])

    delta3 = d_sigmoid(z3) * d_c
    a3_n = W3.T @ delta3
    W3_N = W3 - (learning_rate * np.outer(delta3, a2))
    b3_n = b3 - (learning_rate * delta3)

    delta2 = d_sigmoid(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_N = W2 - (learning_rate * np.outer(delta2, a1))
    b2_n = b2 - (learning_rate * delta2)

    delta1 = d_sigmoid(z1) * a2_n
    W1_N = W1 - (learning_rate * np.outer(delta1, input_data))
    b1_n = b1 - (learning_rate * delta1)

    return ([W1_N, W2_N, W3_N], [b1_n, b2_n, b3_n])

In [ ]:
def neural_step(
    input_data: np.ndarray,
    label: int,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    weights = weights.copy()
    biases = biases.copy()

    a = [input_data]
    z: list[np.ndarray] = []
    for W, b in zip(weights, biases):
        z.append(W @ a[-1] + b)
        a.append(sigmoid(z[-1]))

    delta_a = 2 * (a[-1] - np.eye(10)[label]) # d_c

    for i in reversed(range(len(weights))):
        delta = d_sigmoid(z[i]) * delta_a
        delta_a = weights[i].T @ delta

        weights[i] = weights[i] - (learning_rate * np.outer(delta, a[i]))
        biases[i] = biases[i] - (learning_rate * delta)

    return (weights, biases)

In [6]:
W1 = np.random.randn(16, 784) * 0.01
W2 = np.random.randn(16, 16) * 0.01
W3 = np.random.randn(10, 16) * 0.01

b1 = np.zeros(16)
b2 = np.zeros(16)
b3 = np.zeros(10)

for i in range(len(images_scaled)):
    (W1, W2, W3), (b1, b2, b3) = neural_step(images_scaled[i], labels[i], 
                [W1, W2, W3],
                [b1, b2, b3],
                .1)


In [7]:
def classify_image(image: np.ndarray) -> int:
    z1 = W1 @ image + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)
    z3 = W3 @ a2 + b3
    a3 = sigmoid(z3)

    return np.argmax(a3).astype(int)

In [8]:
test_image_count = 4

print(classify_image(images_scaled[test_image_count]))
print(labels[test_image_count])

9
9


In [9]:
output = np.array([])

for i in range(len(test_images_scaled)):
    output = np.append(output, classify_image(test_images_scaled[i]))

In [10]:
print(output[:10].astype(int))
print(test_labels[:10])

[7 2 1 0 4 1 4 9 6 9]
[7 2 1 0 4 1 4 9 5 9]


In [11]:
matches = output == test_labels
correct_count = np.sum(matches)
total_count = len(test_labels)

print(f"Correct predictions: {correct_count} out of {total_count}")
print(f"Accuracy: {correct_count / total_count * 100:.2f}%")

plain_accuracy = correct_count / total_count * 100

Correct predictions: 7971 out of 10000
Accuracy: 79.71%


# Test

## Which Number Perform Best and Worst

In [77]:
results = {x: 0 for x in range(10)}
image_count = {x: 0 for x in range(10)}

for i in range(len(test_images_scaled)):
    predicted_label = output[i]
    if predicted_label == test_labels[i]:
        results[predicted_label] += 1

    image_count[test_labels[i]] += 1


print(results)
print(image_count)
accuracy = {x: round(results[x] / image_count[x] * 100, 2) for x in range(10)}
print(accuracy)

{0: 888, 1: 1101, 2: 779, 3: 922, 4: 858, 5: 57, 6: 852, 7: 769, 8: 739, 9: 888}
{0: 980, 1: 1135, 2: 1032, 3: 1010, 4: 982, 5: 892, 6: 958, 7: 1028, 8: 974, 9: 1009}
{0: 90.61, 1: 97.0, 2: 75.48, 3: 91.29, 4: 87.37, 5: 6.39, 6: 88.94, 7: 74.81, 8: 75.87, 9: 88.01}


In [78]:
df = pd.DataFrame(accuracy.items(), columns=["Digit", "Accuracy"])


# 2. Create the Bar Chart
fig = px.bar(
    df,
    x="Digit",
    y="Accuracy",
    title="Neural Network Performance by Digit",
    text="Accuracy",  # Shows the number on top of the bar
    color="Accuracy",  # Colors bars based on performance
    color_continuous_scale="RdYlGn",  # Red-Yellow-Green scale
)

# 3. Clean up the look
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105], xaxis_title="Digit", yaxis_title="Accuracy (%)")
fig.show()

In [26]:
# https://stanford.edu/~shervine/teaching/cs-229/cheatsheet-machine-learning-tips-and-tricks/#classification-metrics

CM = np.zeros((10, 10), dtype=int)

for i in range(len(test_images_scaled)):
    predicted_label = output[i].astype(int)
    correct_label = test_labels[i]
    CM[correct_label, predicted_label] += 1
    # so for the 1 row it shows the number of times each one image was classified as in 1 - 10 col

fig = px.imshow(
    CM,
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=[str(i) for i in range(10)],
    y=[str(i) for i in range(10)],
)
fig.show()

In [27]:
TP = np.diag(CM)
FP = np.sum(CM, axis=0) - TP
FN = np.sum(CM, axis=1) - TP
TN = np.sum(CM) - (FP + FN + TP)

global_accuracy = np.sum(TP) / np.sum(CM)
precision = TP / (TP + FP)
recall = TP / (TP + FN) # how good at finding the target
specificity = TN / (TN + FP) # how good at avoiding non targets

print(
    f"Digit 5 - Precision: {precision[5]:.2f}, Recall: {recall[5]:.2f}, Specificity: {specificity[5]:.2f}"
)

Digit 5 - Precision: 0.85, Recall: 0.51, Specificity: 0.99


In [28]:
digits = [str(i) for i in range(10)]

fig = go.Figure()
fig.add_trace(go.Bar(x=digits, y=precision, name="Precision (Trust)"))
fig.add_trace(go.Bar(x=digits, y=recall, name="Recall (Coverage)"))

fig.update_layout(
    title="Precision vs Recall per Digit",
    barmode="group",
    yaxis_range=[0, 1.1],
    xaxis_title="Digit",
    yaxis_title="Score (0-1)",
)
fig.show()

In [29]:
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
df_f1 = pd.DataFrame(zip(range(10), f1_scores.round(2)), columns=["Digit", "F1-Score"])

fig = px.bar(
    df_f1,
    x="Digit",
    y="F1-Score",
    title="Neural Network Performance by Digit and F1 Score",
    text="F1-Score",  # Shows the number on top of the bar
    color="F1-Score",  # Colors bars based on performance
    color_continuous_scale="RdYlGn",  # Red-Yellow-Green scale
)

fig.show()

## Different Learning Rates

In [30]:
def neural_step_with_cost(
    input_data: np.ndarray,
    label: int,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)
    z3 = W3 @ a2 + b3
    a3 = sigmoid(z3)

    c = 0.5 * np.sum((a3 - np.eye(10)[label]) ** 2)
    d_c = 2 * (a3 - np.eye(10)[label])

    delta3 = d_sigmoid(z3) * d_c
    a3_n = W3.T @ delta3
    W3_N = W3 - (learning_rate * np.outer(delta3, a2))
    b3_n = b3 - (learning_rate * delta3)

    delta2 = d_sigmoid(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_N = W2 - (learning_rate * np.outer(delta2, a1))
    b2_n = b2 - (learning_rate * delta2)

    delta1 = d_sigmoid(z1) * a2_n
    W1_N = W1 - (learning_rate * np.outer(delta1, input_data))
    b1_n = b1 - (learning_rate * delta1)

    return ([W1_N, W2_N, W3_N], [b1_n, b2_n, b3_n], c)

In [ ]:
learning_rates = [10.0, 1.0, .1, .01, .001]

results_learning_rates: dict[float, list[float]] = {}
accuracy: dict[float, float] = {}

W1_LEARN = np.random.randn(16, 784) * 0.01
W2_LEARN = np.random.randn(16, 16) * 0.01
W3_LEARN = np.random.randn(10, 16) * 0.01

b1_LEARN = np.zeros(16)
b2_LEARN = np.zeros(16)
b3_LEARN = np.zeros(10)


for lr in learning_rates:
    (W1_TEST, W2_TEST, W3_TEST), (b1_TEST, b2_TEST, b3_TEST) = (W1_LEARN.copy(), W2_LEARN.copy(), W3_LEARN.copy()), (b1_LEARN.copy(), b2_LEARN.copy(), b3_LEARN.copy())
    results_learning_rates[lr] = []

    for i in range(len(images_scaled)):
        (W1_TEST, W2_TEST, W3_TEST), (b1_TEST, b2_TEST, b3_TEST), cost = neural_step_with_cost(images_scaled[i], labels[i],
                    [W1_TEST, W2_TEST, W3_TEST],
                    [b1_TEST, b2_TEST, b3_TEST],
                    lr)
        results_learning_rates[lr].append(cost)

    output = np.array([])
    for i in range(len(test_images_scaled)):
        z1 = W1_TEST @ test_images_scaled[i] + b1_TEST
        a1 = sigmoid(z1)
        z2 = W2_TEST @ a1 + b2_TEST
        a2 = sigmoid(z2)
        z3 = W3_TEST @ a2 + b3_TEST
        a3 = sigmoid(z3)
        output = np.append(output, np.argmax(a3).astype(int))

    matches = output == test_labels
    accuracy[lr] = np.sum(matches) / len(test_labels) * 100

In [69]:
fig = go.Figure()
for lr, loss_history in results_learning_rates.items():
    fig.add_trace(go.Scatter(y=loss_history[::400], name=f"LR: {lr}"))

fig.update_layout(
    title="Learning Rate Comparison", xaxis_title="Iteration", yaxis_title="Loss"
)
fig.show()

accuracy

{10.0: np.float64(8.92),
 1.0: np.float64(80.97),
 0.1: np.float64(65.41),
 0.01: np.float64(10.280000000000001),
 0.001: np.float64(11.35)}

In [70]:
import pandas as pd

fig = go.Figure()
for lr, loss_history in results_learning_rates.items():
    # Convert to a Series and calculate a rolling average of 500 points
    smooth_loss = pd.Series(loss_history).rolling(window=500).mean()

    fig.add_trace(go.Scatter(y=smooth_loss, name=f"LR: {lr}", opacity=0.8))

fig.update_layout(title="Smoothed Learning Rate Comparison (Moving Average)")
fig.show()

## Hidden Layer Pics

In [83]:
W2.shape

(16, 16)

In [ ]:
fig = make_subplots(rows=4, cols=4, 
                        subplot_titles=[f"Neuron {i}" for i in range(16)])

for i in range(16):
    neuron_weights = W1[i].reshape(28, 28)
    
    trace = go.Heatmap(
        z=neuron_weights, 
        colorscale='RdBu',
        zmid=0,    
        showscale=False
    )
    
    row = (i // 4) + 1
    col = (i % 4) + 1
    fig.add_trace(trace, row=row, col=col)

fig.update_layout(height=800, width=800, title_text="Hidden Layer 1 Weights")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange='reversed')
fig.show()

In [109]:
image_index = 3

test_img = test_images_scaled[image_index]
test_labels[image_index]

a1_test = W1 @ test_img + b1

In [110]:
fig = make_subplots(rows=4, cols=4, 
                        subplot_titles=[f"Act: {val:.2f}" for val in a1_test])

for i in range(16):
    # Multiply the 'Weights' by the 'Activation'
    # This "dims" the neurons that didn't fire
    weighted_neuron = W1[i].reshape(28, 28) * a1_test[i]
    
    trace = go.Heatmap(
        z=weighted_neuron, 
        colorscale='RdBu', # Red-Blue shows pos/neg clearly
        zmid=0,            # Force 0 to be the middle color
        showscale=False
    )
    
    row = (i // 4) + 1
    col = (i % 4) + 1
    fig.add_trace(trace, row=row, col=col)

    
fig.update_layout(
    height=800, width=800, title_text="What the Hidden Layer 'Saw' in this Image"
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()

I was using 0 as an example and we see how the activation for the many of the ones with strong lines close to a 1 are activated negatively and we see how it's red

## Preprocessing with PCA and SVD

So PCA is finding the eigenvectors of the covariance matrix which here is images - avg. Once we have the covariance we can just find SVD which sorts all the eigenvectors by importance. Then we can chop of the smallest bottom of the matrix of eigenvals V. Then run the smaller principal components through the neural network.

In [227]:
pca_mnist_avg = np.average(images_scaled, axis=0)
pca_images_shifted = images_scaled - pca_mnist_avg

In [228]:
U, S, Vt = la.svd(pca_images_shifted, full_matrices=False)
V = Vt.T

In [230]:
reduced_dimensions = 100
reduced_V = V[:, :reduced_dimensions]

pca_images_projected = pca_images_shifted @ reduced_V
pca_test_images_projected = (test_images_scaled - pca_mnist_avg) @ reduced_V

In [163]:
fig = make_subplots(rows=4, cols=4)

for i in range(16):
    # Reshape the i-th column of V
    component_img = V[:, i].reshape(28, 28)

    fig.add_trace(
        go.Heatmap(z=component_img, colorscale="RdBu", zmid=0, showscale=False),
        row=(i // 4) + 1,
        col=(i % 4) + 1,
    )

fig.update_layout(height=800, width=800, title_text="The 16 Most Important Principal Components")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")


fig.show()

In [310]:
W1_PCA = np.random.randn(16, reduced_dimensions) * np.sqrt(1 / reduced_dimensions)
W2_PCA = np.random.randn(16, 16) * np.sqrt(1 / 16)
W3_PCA = np.random.randn(10, 16) * np.sqrt(1 / 16)


b1_pca = np.zeros(16)
b2_pca = np.zeros(16)
b3_pca = np.zeros(10)

In [397]:
for i in range(len(images_scaled)):
    (W1_PCA, W2_PCA, W3_PCA), (b1_pca, b2_pca, b3_pca) = neural_step(
        pca_images_projected[i],
        labels[i],
        [W1_PCA, W2_PCA, W3_PCA],
        [b1_pca, b2_pca, b3_pca],
        0.1,
    )

In [398]:
output = np.array([])

for i in range(len(pca_test_images_projected)):
    z1 = W1_PCA @ pca_test_images_projected[i] + b1_pca
    a1 = sigmoid(z1)
    z2 = W2_PCA @ a1 + b2_pca
    a2 = sigmoid(z2)
    z3 = W3_PCA @ a2 + b3_pca
    a3 = sigmoid(z3)
    output = np.append(output, np.argmax(a3).astype(int))

In [399]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(92.78999999999999)

This actually ends up getting to the same precision above 80 percent every time I run no matter the randomness unlike the normal one

## Xavier or backprop stays alive 

In [40]:
W1 = np.random.randn(16, 784) * np.sqrt(1 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(1 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(1 / 16)


b1 = np.zeros(16)
b2 = np.zeros(16)
b3 = np.zeros(10)

In [54]:
for i in range(len(images_scaled)):
    (W1, W2, W3), (b1, b2, b3) = neural_step(
        images_scaled[i],
        labels[i],
        [W1, W2, W3],
        [b1, b2, b3],
        .1,
    )

In [55]:
output = np.array([])

for i in range(len(test_images_scaled)):
    z1 = W1 @ test_images_scaled[i] + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)
    z3 = W3 @ a2 + b3
    a3 = sigmoid(z3)
    output = np.append(output, np.argmax(a3).astype(int))

In [56]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(88.33)

## Test Relu

In [16]:
def ReLU(x: np.ndarray) -> np.ndarray:
    return np.maximum(0, x)

def d_ReLU(x: np.ndarray) -> np.ndarray:
    return np.where(x > 0, 1, 0)


In [61]:
def neural_step_relu(
    input_data: np.ndarray,
    label: int,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    a3 = ReLU(z3)

    d_c = 2 * (a3 - np.eye(10)[label])

    delta3 = d_ReLU(z3) * d_c
    a3_n = W3.T @ delta3
    W3_N = W3 - (learning_rate * np.outer(delta3, a2))
    b3_n = b3 - (learning_rate * delta3)

    delta2 = d_ReLU(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_N = W2 - (learning_rate * np.outer(delta2, a1))
    b2_n = b2 - (learning_rate * delta2)

    delta1 = d_ReLU(z1) * a2_n
    W1_N = W1 - (learning_rate * np.outer(delta1, input_data))
    b1_n = b1 - (learning_rate * delta1)

    return ([W1_N, W2_N, W3_N], [b1_n, b2_n, b3_n])

In [82]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros(16)
b2 = np.zeros(16)
b3 = np.zeros(10)

In [83]:
for i in range(len(images_scaled)):
    (W1, W2, W3), (b1, b2, b3) = neural_step_relu(
        images_scaled[i],
        labels[i],
        [W1, W2, W3],
        [b1, b2, b3],
        0.001,
    )

In [84]:
output = np.array([])

for i in range(len(test_images_scaled)):
    z1 = W1 @ test_images_scaled[i] + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    a3 = ReLU(z3)
    output = np.append(output, np.argmax(a3).astype(int))

In [85]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(83.28999999999999)

## Relu with Softmax

When we get to the last layer to compute the cost we end up having something like for 22 or random big numbers. But we end up having the actual answer be 1. So the cost will be high which is probably why I had to do a learning rate of .001 to compensate. But instead if I use softmax which gives probability similar to the 1 probablity or 100 percent chance it is that number.

In [46]:
def softmax(x: np.ndarray) -> np.ndarray:
    # It's just e/e_sum but we do x-max for numerical stability
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)


def delta_softmax(a3, d_c) -> np.ndarray:
    # Bruh since the derivative is related to each other value the solution is a partial derivative with jacobian
    jacobian = np.diag(a3) - np.outer(a3, a3)
    # Dot product the error with the Jacobian to get the delta
    return jacobian @ d_c

In [68]:
def neural_step_softmax(
    input_data: np.ndarray,
    label: int,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    a3 = softmax(z3)

    d_c = 2 * (a3 - np.eye(10)[label])

    delta3 = delta_softmax(a3, d_c)
    a3_n = W3.T @ delta3
    W3_N = W3 - (learning_rate * np.outer(delta3, a2))
    b3_n = b3 - (learning_rate * delta3)

    delta2 = d_ReLU(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_N = W2 - (learning_rate * np.outer(delta2, a1))
    b2_n = b2 - (learning_rate * delta2)

    delta1 = d_ReLU(z1) * a2_n
    W1_N = W1 - (learning_rate * np.outer(delta1, input_data))
    b1_n = b1 - (learning_rate * delta1)

    return ([W1_N, W2_N, W3_N], [b1_n, b2_n, b3_n])

In [88]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros(16)
b2 = np.zeros(16)
b3 = np.zeros(10)

In [89]:
for i in range(len(images_scaled)):
    (W1, W2, W3), (b1, b2, b3) = neural_step_softmax(
        images_scaled[i],
        labels[i],
        [W1, W2, W3],
        [b1, b2, b3],
        0.01,
    )

In [90]:
output = np.array([])

for i in range(len(test_images_scaled)):
    z1 = W1 @ test_images_scaled[i] + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    # a3 = softmax(z3) - don't need basically probability
    output = np.append(output, np.argmax(z3).astype(int))

In [91]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(91.56)

## Cross Entropy

Also if I use softmax it'll simplify with cross entropy to have final delta just be like mean squared.
It's more logarithmic than just the dist between correct and guess.

In [141]:
def neural_step_cross_entropy(
    input_data: np.ndarray,
    label: int,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    a3 = softmax(z3)

    delta3 = a3 - np.eye(10)[label]
    a3_n = W3.T @ delta3
    W3_N = W3 - (learning_rate * np.outer(delta3, a2))
    b3_n = b3 - (learning_rate * delta3)

    delta2 = d_ReLU(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_N = W2 - (learning_rate * np.outer(delta2, a1))
    b2_n = b2 - (learning_rate * delta2)

    delta1 = d_ReLU(z1) * a2_n
    W1_N = W1 - (learning_rate * np.outer(delta1, input_data))
    b1_n = b1 - (learning_rate * delta1)

    return ([W1_N, W2_N, W3_N], [b1_n, b2_n, b3_n])

In [142]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros(16)
b2 = np.zeros(16)
b3 = np.zeros(10)

In [146]:
for i in range(len(images_scaled)):
    (W1, W2, W3), (b1, b2, b3) = neural_step_cross_entropy(
        images_scaled[i],
        labels[i],
        [W1, W2, W3],
        [b1, b2, b3],
        0.01,
    )

In [147]:
output = np.array([])

for i in range(len(test_images_scaled)):
    z1 = W1 @ test_images_scaled[i] + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    # a3 = softmax(z3) - don't need basically probability
    output = np.append(output, np.argmax(z3).astype(int))

In [148]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(93.07)

## Mini Batch - Schocastic Things

In [161]:
def neural_step_batch(
    input_data: np.ndarray,
    labels: np.ndarray,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data.T + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    a3 = softmax(z3)

    delta3 = a3 - np.eye(10)[labels].T
    a3_n = W3.T @ delta3
    W3_N = W3 - ((learning_rate / input_data.shape[0]) * (delta3 @ a2.T))
    b3_n = b3 - ((learning_rate / input_data.shape[0]) * np.sum(delta3, axis=1, keepdims=True))

    delta2 = d_ReLU(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_N = W2 - ((learning_rate / input_data.shape[0]) * (delta2 @ a1.T))
    b2_n = b2 - ((learning_rate / input_data.shape[0]) * np.sum(delta2, axis=1, keepdims=True))

    delta1 = d_ReLU(z1) * a2_n
    W1_N = W1 - ((learning_rate / input_data.shape[0]) * (delta1 @ input_data))
    b1_n = b1 - ((learning_rate / input_data.shape[0]) * np.sum(delta1, axis=1, keepdims=True))

    return ([W1_N, W2_N, W3_N], [b1_n, b2_n, b3_n])

In [238]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros((16, 1))
b2 = np.zeros((16, 1))
b3 = np.zeros((10, 1))

batch_size = 64

In [293]:
for i in range(len(images_scaled) // batch_size):
    (W1, W2, W3), (b1, b2, b3) = neural_step_batch(
        images_scaled[i * batch_size : (i + 1) * batch_size],
        labels[i * batch_size : (i + 1) * batch_size],
        [W1, W2, W3],
        [b1, b2, b3],
        .05,
    )

In [294]:
Z1 = W1 @ test_images_scaled.T + b1.reshape(-1, 1)
A1 = ReLU(Z1)
Z2 = W2 @ A1 + b2.reshape(-1, 1)
A2 = ReLU(Z2)
Z3 = W3 @ A2 + b3.reshape(-1, 1)


output = np.argmax(Z3, axis=0)

In [295]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(95.08)

## Rate Decay During Multiple Epochs

In [323]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros((16, 1))
b2 = np.zeros((16, 1))
b3 = np.zeros((10, 1))

batch_size = 64
epochs = 20
learning_rate = .1

In [324]:
for j in range(epochs):
    for i in range(len(images_scaled) // batch_size):
        (W1, W2, W3), (b1, b2, b3) = neural_step_batch(
            images_scaled[i * batch_size : (i + 1) * batch_size],
            labels[i * batch_size : (i + 1) * batch_size],
            [W1, W2, W3],
            [b1, b2, b3],
            learning_rate,
        )

In [325]:
Z1 = W1 @ test_images_scaled.T + b1.reshape(-1, 1)
A1 = ReLU(Z1)
Z2 = W2 @ A1 + b2.reshape(-1, 1)
A2 = ReLU(Z2)
Z3 = W3 @ A2 + b3.reshape(-1, 1)


output = np.argmax(Z3, axis=0)

In [326]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(95.0)

Spending more epochs actually gives a better result

In [453]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros((16, 1))
b2 = np.zeros((16, 1))
b3 = np.zeros((10, 1))

batch_size = 64
epochs = 20
learning_rate = .1

In [454]:
for j in range(epochs):
    for i in range(len(images_scaled) // batch_size):
        (W1, W2, W3), (b1, b2, b3) = neural_step_batch(
            images_scaled[i * batch_size : (i + 1) * batch_size],
            labels[i * batch_size : (i + 1) * batch_size],
            [W1, W2, W3],
            [b1, b2, b3],
            learning_rate,
        )
    learning_rate *= 0.9

In [455]:
Z1 = W1 @ test_images_scaled.T + b1.reshape(-1, 1)
A1 = ReLU(Z1)
Z2 = W2 @ A1 + b2.reshape(-1, 1)
A2 = ReLU(Z2)
Z3 = W3 @ A2 + b3.reshape(-1, 1)

output = np.argmax(Z3, axis=0)

In [456]:
matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(95.87)

### Momentum

In [400]:
def neural_step_batch_update(
    input_data: np.ndarray,
    labels: np.ndarray,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    W1, W2, W3 = weights
    b1, b2, b3 = biases

    z1 = W1 @ input_data.T + b1
    a1 = ReLU(z1)
    z2 = W2 @ a1 + b2
    a2 = ReLU(z2)
    z3 = W3 @ a2 + b3
    a3 = softmax(z3)

    delta3 = a3 - np.eye(10)[labels].T
    a3_n = W3.T @ delta3
    W3_update = (learning_rate / input_data.shape[0]) * (delta3 @ a2.T)
    b3_n_update = (learning_rate / input_data.shape[0]) * np.sum(
        delta3, axis=1, keepdims=True
    )

    delta2 = d_ReLU(z2) * a3_n
    a2_n = W2.T @ delta2
    W2_update = (learning_rate / input_data.shape[0]) * (delta2 @ a1.T)
    b2_n_update = (learning_rate / input_data.shape[0]) * np.sum(
        delta2, axis=1, keepdims=True
    )

    delta1 = d_ReLU(z1) * a2_n
    W1_update = (learning_rate / input_data.shape[0]) * (delta1 @ input_data)
    b1_n_update = (learning_rate / input_data.shape[0]) * np.sum(
        delta1, axis=1, keepdims=True
    )

    return ([W1_update, W2_update, W3_update], [b1_n_update, b2_n_update, b3_n_update])

In [460]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros((16, 1))
b2 = np.zeros((16, 1))
b3 = np.zeros((10, 1))

batch_size = 64
epochs = 7
learning_rate = 0.01

In [461]:
W1_vel, W2_vel, W3_vel = np.zeros_like(W1), np.zeros_like(W2), np.zeros_like(W3)
b1_vel, b2_vel, b3_vel = np.zeros_like(b1), np.zeros_like(b2), np.zeros_like(b3)

beta = 0.9 # Friction

for j in range(epochs):
    for i in range(len(images_scaled) // batch_size):
        (W1_update, W2_update, W3_update), (b1_n_update, b2_n_update, b3_n_update) = (
            neural_step_batch_update(
                images_scaled[i * batch_size : (i + 1) * batch_size],
                labels[i * batch_size : (i + 1) * batch_size],
                [W1, W2, W3],
                [b1, b2, b3],
                learning_rate,
            )
        )

        W1_vel = (beta * W1_vel) + W1_update
        W2_vel = (beta * W2_vel) + W2_update
        W3_vel = (beta * W3_vel) + W3_update

        b1_vel = (beta * b1_vel) + b1_n_update
        b2_vel = (beta * b2_vel) + b2_n_update
        b3_vel = (beta * b3_vel) + b3_n_update

        W1 -= W1_vel
        W2 -= W2_vel
        W3 -= W3_vel

        b1 -= b1_vel
        b2 -= b2_vel
        b3 -= b3_vel

    learning_rate *= 0.90

In [462]:
Z1 = W1 @ test_images_scaled.T + b1.reshape(-1, 1)
A1 = ReLU(Z1)
Z2 = W2 @ A1 + b2.reshape(-1, 1)
A2 = ReLU(Z2)
Z3 = W3 @ A2 + b3.reshape(-1, 1)

output = np.argmax(Z3, axis=0)

matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(95.07)

Due to the momentum it gets closer to the 95 in even 7 epochs compared to the version before

### Neuron Size Changes

4th layer?

In [493]:
def neural_step_batch_update_any(
    input_data: np.ndarray,
    labels: np.ndarray,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    batch_size = input_data.shape[0]

    a = [input_data.T]
    z: list[np.ndarray] = []
    for i in range(len(weights)):
        z.append(weights[i] @ a[-1] + biases[i])
        if i == len(weights) - 1:
            a.append(softmax(z[-1]))
        else:
            a.append(ReLU(z[-1]))

    w_updates = [None] * len(weights)
    b_updates = [None] * len(biases)
    delta = a[-1] - np.eye(10)[labels].T

    for i in reversed(range(len(weights))):
        w_updates[i] = (learning_rate / batch_size) * (delta @ a[i].T)
        b_updates[i] = (learning_rate / batch_size) * np.sum(delta, axis=1, keepdims=True)

        if i > 0:
            delta = (weights[i].T @ delta) * d_ReLU(z[i - 1])

    return (w_updates, b_updates)

In [517]:
W1 = np.random.randn(16, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W3 = np.random.randn(16, 16) * np.sqrt(2 / 16)
W4 = np.random.randn(10, 16) * np.sqrt(2 / 16)


b1 = np.zeros((16, 1))
b2 = np.zeros((16, 1))
b3 = np.zeros((16, 1))
b4 = np.zeros((10, 1))

batch_size = 64
epochs = 20
learning_rate = 0.01

In [518]:
W1_vel, W2_vel, W3_vel, W4_vel = np.zeros_like(W1), np.zeros_like(W2), np.zeros_like(W3), np.zeros_like(W4)
b1_vel, b2_vel, b3_vel, b4_vel = np.zeros_like(b1), np.zeros_like(b2), np.zeros_like(b3), np.zeros_like(b4)

beta = 0.9  # Friction

for j in range(epochs):
    for i in range(len(images_scaled) // batch_size):
        (W1_update, W2_update, W3_update, W4_update), (
            b1_n_update,
            b2_n_update,
            b3_n_update,
            b4_n_update,
        ) = neural_step_batch_update_any(
            images_scaled[i * batch_size : (i + 1) * batch_size],
            labels[i * batch_size : (i + 1) * batch_size],
            [W1, W2, W3, W4],
            [b1, b2, b3, b4],
            learning_rate,
        )

        W1_vel = (beta * W1_vel) + W1_update
        W2_vel = (beta * W2_vel) + W2_update
        W3_vel = (beta * W3_vel) + W3_update
        W4_vel = (beta * W4_vel) + W4_update

        b1_vel = (beta * b1_vel) + b1_n_update
        b2_vel = (beta * b2_vel) + b2_n_update
        b3_vel = (beta * b3_vel) + b3_n_update
        b4_vel = (beta * b4_vel) + b4_n_update

        W1 -= W1_vel
        W2 -= W2_vel
        W3 -= W3_vel
        W4 -= W4_vel

        b1 -= b1_vel
        b2 -= b2_vel
        b3 -= b3_vel
        b4 -= b4_vel

    learning_rate *= 0.90

In [519]:
Z1 = W1 @ test_images_scaled.T + b1.reshape(-1, 1)
A1 = ReLU(Z1)
Z2 = W2 @ A1 + b2.reshape(-1, 1)
A2 = ReLU(Z2)
Z3 = W3 @ A2 + b3.reshape(-1, 1)
A3 = ReLU(Z3)
Z4 = W4 @ A3 + b4.reshape(-1, 1)

output = np.argmax(Z4, axis=0)

matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(95.21)

What about 64, 32, 10

In [520]:
W1 = np.random.randn(64, 784) * np.sqrt(2 / 784)
W2 = np.random.randn(32, 64) * np.sqrt(2 / 64)
W3 = np.random.randn(10, 32) * np.sqrt(2 / 32)


b1 = np.zeros((64, 1))
b2 = np.zeros((32, 1))
b3 = np.zeros((10, 1))

batch_size = 64
epochs = 20
learning_rate = 0.01

In [521]:
W1_vel, W2_vel, W3_vel = np.zeros_like(W1), np.zeros_like(W2), np.zeros_like(W3)
b1_vel, b2_vel, b3_vel = np.zeros_like(b1), np.zeros_like(b2), np.zeros_like(b3)

beta = 0.9  # Friction

for j in range(epochs):
    for i in range(len(images_scaled) // batch_size):
        (W1_update, W2_update, W3_update), (b1_n_update, b2_n_update, b3_n_update) = (
            neural_step_batch_update_any(
                images_scaled[i * batch_size : (i + 1) * batch_size],
                labels[i * batch_size : (i + 1) * batch_size],
                [W1, W2, W3],
                [b1, b2, b3],
                learning_rate,
            )
        )

        W1_vel = (beta * W1_vel) + W1_update
        W2_vel = (beta * W2_vel) + W2_update
        W3_vel = (beta * W3_vel) + W3_update

        b1_vel = (beta * b1_vel) + b1_n_update
        b2_vel = (beta * b2_vel) + b2_n_update
        b3_vel = (beta * b3_vel) + b3_n_update

        W1 -= W1_vel
        W2 -= W2_vel
        W3 -= W3_vel

        b1 -= b1_vel
        b2 -= b2_vel
        b3 -= b3_vel

    learning_rate *= 0.90

In [522]:
Z1 = W1 @ test_images_scaled.T + b1.reshape(-1, 1)
A1 = ReLU(Z1)
Z2 = W2 @ A1 + b2.reshape(-1, 1)
A2 = ReLU(Z2)
Z3 = W3 @ A2 + b3.reshape(-1, 1)

output = np.argmax(Z3, axis=0)

matches = output == test_labels
np.sum(matches) / len(test_labels) * 100

np.float64(97.39)

64, 32, 16, 10?

In [677]:
layers = [784, 128, 64, 10]

weights = [
    np.random.randn(layers[i + 1], layers[i]) * np.sqrt(2 / layers[i])
    for i in range(len(layers) - 1)
]
biases = [np.zeros((layers[i + 1], 1)) for i in range(len(layers) - 1)]

w_vel = [np.zeros_like(w) for w in weights]
b_vel = [np.zeros_like(b) for b in biases]

batch_size = 64
epochs = 20
learning_rate = 0.01
beta = 0.9

In [678]:
for j in range(epochs):
    for i in range(len(images_scaled) // batch_size):
        w_updates, b_updates = neural_step_batch_update_any(
            images_scaled[i * batch_size : (i + 1) * batch_size],
            labels[i * batch_size : (i + 1) * batch_size],
            weights,
            biases,
            learning_rate,
        )

        for k in range(len(weights)):
            w_vel[k] = (beta * w_vel[k]) + w_updates[k]
            b_vel[k] = (beta * b_vel[k]) + b_updates[k]
            
            weights[k] -= w_vel[k]
            biases[k] -= b_vel[k]

    learning_rate *= 0.95

In [679]:
a_test = test_images_scaled.T

for i in range(len(weights) - 1):
    z_test = weights[i] @ a_test + biases[i]
    a_test = ReLU(z_test)

z_final = weights[-1] @ a_test + biases[-1]
output = np.argmax(z_final, axis=0)

accuracy = np.mean(output == test_labels) * 100
print(f"Final Test Accuracy: {accuracy:.2f}%")

Final Test Accuracy: 97.80%


### Randomnes??

In [583]:
layers = [784, 128, 32, 16, 10]

weights = [
    np.random.randn(layers[i + 1], layers[i]) * np.sqrt(2 / layers[i])
    for i in range(len(layers) - 1)
]
biases = [np.zeros((layers[i + 1], 1)) for i in range(len(layers) - 1)]

w_vel = [np.zeros_like(w) for w in weights]
b_vel = [np.zeros_like(b) for b in biases]

batch_size = 64
epochs = 40
learning_rate = 0.01
beta = 0.9

In [584]:
for j in range(epochs):
    indices = np.random.permutation(len(images_scaled))
    X_shuffled = images_scaled[indices]
    Y_shuffled = labels[indices]

    for i in range(len(images_scaled) // batch_size):
        batch_X = X_shuffled[i * batch_size : (i + 1) * batch_size]
        batch_Y = Y_shuffled[i * batch_size : (i + 1) * batch_size]
        
        w_updates, b_updates = neural_step_batch_update_any(
            batch_X,
            batch_Y,
            weights,
            biases,
            learning_rate,
        )

        for k in range(len(weights)):
            w_vel[k] = (beta * w_vel[k]) + w_updates[k]
            b_vel[k] = (beta * b_vel[k]) + b_updates[k]

            weights[k] -= w_vel[k]
            biases[k] -= b_vel[k]

    learning_rate *= 0.90

In [585]:
a_test = test_images_scaled.T

for i in range(len(weights) - 1):
    z_test = weights[i] @ a_test + biases[i]
    a_test = ReLU(z_test)

z_final = weights[-1] @ a_test + biases[-1]
output = np.argmax(z_final, axis=0)

accuracy = np.mean(output == test_labels) * 100
print(f"Final Test Accuracy: {accuracy:.2f}%")

Final Test Accuracy: 97.77%


doesn't really work I don't think model was big enough where we have it memorizing order of inputs

### Data Augmentation

In [636]:
def rotate_batch(image_batch_flat: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w = 28, 28

    # Take all 784 points and each point is like a vector and forming a big matrix multiply to the rotation matrix
    y, x = np.indices((h, w))
    half_h, half_w = h // 2, w // 2
    points = np.stack([x.flatten() - half_w, y.flatten() - half_h])

    theta = np.radians(angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    rot_mat = np.array([[c, s], [-s, c]])

    new_points = rot_mat @ points

    new_x = (new_points[0] + half_w).astype(int)
    new_y = (new_points[1] + half_h).astype(int)

    # Take out points outside image
    mask = (new_x >= 0) & (new_x < w) & (new_y >= 0) & (new_y < h)

    rotated_batch = np.zeros_like(image_batch_flat)

    target_indices = new_y[mask] * 28 + new_x[mask]
    source_indices = y.flatten()[mask] * 28 + x.flatten()[mask]

    rotated_batch[:, target_indices] = image_batch_flat[:, source_indices]

    return rotated_batch

In [666]:
def rotate_batch_inverse_fast(image_batch, angle_deg):
    image_batch = np.asarray(image_batch)
    batch_size = image_batch.shape[0]

    # 1. Target Grid (The empty seats we want to fill)
    y, x = np.indices((28, 28))
    target_points = np.stack([x.flatten() - 14, y.flatten() - 14])

    # 2. Inverse Rotation (The "Where did I come from?" math)
    theta = np.radians(-angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    inv_R = np.array([[c, -s], [s, c]])

    src_points = inv_R @ target_points
    src_x = np.round(src_points[0] + 14).astype(int)
    src_y = np.round(src_points[1] + 14).astype(int)

    # 3. Mask
    mask = (src_x >= 0) & (src_x < 28) & (src_y >= 0) & (src_y < 28)

    # 4. Fill (No holes!)
    img_2d = image_batch.reshape(-1, 28, 28)
    output = np.zeros_like(img_2d)

    # Target (y,x) gets color from Source (src_y, src_x)
    output[:, y.flatten()[mask], x.flatten()[mask]] = img_2d[
        :, src_y[mask], src_x[mask]
    ]

    return output.reshape(batch_size, 784)

In [691]:
def transform_batch(image_batch, angle_deg, shift_x, shift_y):
    image_batch = np.asarray(image_batch)
    batch_size = image_batch.shape[0]

    y, x = np.indices((28, 28))
    # 1. Start with the target grid (centered)
    target_points = np.stack([x.flatten() - 14, y.flatten() - 14])

    # 2. Inverse Rotation
    theta = np.radians(-angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    inv_R = np.array([[c, -s], [s, c]])

    # 3. Calculate Source Points (Rotate THEN subtract the shift)
    # Note: We subtract shift because we are asking "Where did this come from?"
    src_points = inv_R @ target_points
    src_x = np.round(src_points[0] + 14 - shift_x).astype(int)
    src_y = np.round(src_points[1] + 14 - shift_y).astype(int)

    mask = (src_x >= 0) & (src_x < 28) & (src_y >= 0) & (src_y < 28)

    img_2d = image_batch.reshape(-1, 28, 28)
    output_2d = np.zeros_like(img_2d)
    output_2d[:, y.flatten()[mask], x.flatten()[mask]] = img_2d[
        :, src_y[mask], src_x[mask]
    ]

    return output_2d.reshape(batch_size, 784)

In [708]:
images_shown = 5
images_to_show = transform_batch(images[:images_shown], 45, 5, 5).reshape(-1, 28, 28)

# 1. Create a subplot grid with 1 row and 'images_shown' columns
fig = make_subplots(
    rows=1,
    cols=images_shown,
    subplot_titles=[f"Label: {l}" for l in labels[:images_shown]],
)

for i in range(images_shown):
    img_trace = go.Heatmap(z=images_to_show[i], colorscale="gray", showscale=False)
    fig.add_trace(img_trace, row=1, col=i + 1)


fig.update_yaxes(autorange="reversed", showticklabels=False)
fig.update_xaxes(showticklabels=False)
fig.update_layout(
    coloraxis_showscale=False,
    height=300,
    width=images_shown * 250,
    margin=dict(l=20, r=20, t=50, b=20),
)

fig.show()

We could blend but we need to do interpolation and blah blah too much work

In [710]:
layers = [784, 128, 64, 10]

weights = [
    np.random.randn(layers[i + 1], layers[i]) * np.sqrt(2 / layers[i])
    for i in range(len(layers) - 1)
]
biases = [np.zeros((layers[i + 1], 1)) for i in range(len(layers) - 1)]

w_vel = [np.zeros_like(w) for w in weights]
b_vel = [np.zeros_like(b) for b in biases]

batch_size = 64
epochs = 20
rotation_epoch = 15
learning_rate = 0.05
beta = 0.9

In [711]:
for j in range(epochs):
    indices = np.random.permutation(len(images_scaled))
    X_shuffled = images_scaled[indices]
    Y_shuffled = labels[indices]

    use_rotation = j < rotation_epoch

    for i in range(len(images_scaled) // batch_size):
        batch_X = X_shuffled[i * batch_size : (i + 1) * batch_size]
        batch_Y = Y_shuffled[i * batch_size : (i + 1) * batch_size]

        if use_rotation:
            random_angle = np.random.uniform(-15, 15)
            sx = np.random.uniform(-2, 2)
            sy = np.random.uniform(-2, 2)
            rotated_batch_X = transform_batch(batch_X, random_angle, sx, sy)
        else:
            rotated_batch_X = batch_X

        w_updates, b_updates = neural_step_batch_update_any(
            rotated_batch_X,
            batch_Y,
            weights,
            biases,
            learning_rate,
        )

        for k in range(len(weights)):
            w_vel[k] = (beta * w_vel[k]) + w_updates[k]
            b_vel[k] = (beta * b_vel[k]) + b_updates[k]

            weights[k] -= w_vel[k]
            biases[k] -= b_vel[k]

    learning_rate *= 0.95

In [712]:
a_test = test_images_scaled.T

for i in range(len(weights) - 1):
    z_test = weights[i] @ a_test + biases[i]
    a_test = ReLU(z_test)

z_final = weights[-1] @ a_test + biases[-1]
output = np.argmax(z_final, axis=0)

accuracy = np.mean(output == test_labels) * 100
print(f"Final Test Accuracy: {accuracy:.2f}%")

Final Test Accuracy: 98.73%


# Final Ideas

In [714]:
def ReLU(x: np.ndarray) -> np.ndarray:
    return np.maximum(0, x)


def d_ReLU(x: np.ndarray) -> np.ndarray:
    return np.where(x > 0, 1, 0)

In [715]:
def softmax(x: np.ndarray) -> np.ndarray:
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

In [716]:
def final_neural_step(
    input_data: np.ndarray,
    labels: np.ndarray,
    weights: list[np.ndarray],
    biases: list[np.ndarray],
    learning_rate: float,
):
    batch_size = input_data.shape[0]

    a = [input_data.T]
    z: list[np.ndarray] = []
    for i in range(len(weights)):
        z.append(weights[i] @ a[-1] + biases[i])
        if i == len(weights) - 1:
            a.append(softmax(z[-1]))
        else:
            a.append(ReLU(z[-1]))

    w_updates = [None] * len(weights)
    b_updates = [None] * len(biases)
    delta = a[-1] - np.eye(10)[labels].T

    for i in reversed(range(len(weights))):
        w_updates[i] = (learning_rate / batch_size) * (delta @ a[i].T)
        b_updates[i] = (learning_rate / batch_size) * np.sum(
            delta, axis=1, keepdims=True
        )

        if i > 0:
            delta = (weights[i].T @ delta) * d_ReLU(z[i - 1])

    return (w_updates, b_updates)

In [717]:
def transform_batch(image_batch, angle_deg, shift_x, shift_y):
    image_batch = np.asarray(image_batch)
    batch_size = image_batch.shape[0]

    y, x = np.indices((28, 28))
    # 1. Start with the target grid (centered)
    target_points = np.stack([x.flatten() - 14, y.flatten() - 14])

    # 2. Inverse Rotation
    theta = np.radians(-angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    inv_R = np.array([[c, -s], [s, c]])

    # 3. Calculate Source Points (Rotate THEN subtract the shift)
    # Note: We subtract shift because we are asking "Where did this come from?"
    src_points = inv_R @ target_points
    src_x = np.round(src_points[0] + 14 - shift_x).astype(int)
    src_y = np.round(src_points[1] + 14 - shift_y).astype(int)

    mask = (src_x >= 0) & (src_x < 28) & (src_y >= 0) & (src_y < 28)

    img_2d = image_batch.reshape(-1, 28, 28)
    output_2d = np.zeros_like(img_2d)
    output_2d[:, y.flatten()[mask], x.flatten()[mask]] = img_2d[
        :, src_y[mask], src_x[mask]
    ]

    return output_2d.reshape(batch_size, 784)

In [730]:
layers = [784, 256, 128, 10]

weights = [
    np.random.randn(layers[i + 1], layers[i]) * np.sqrt(2 / layers[i])
    for i in range(len(layers) - 1)
]
biases = [np.zeros((layers[i + 1], 1)) for i in range(len(layers) - 1)]

w_vel = [np.zeros_like(w) for w in weights]
b_vel = [np.zeros_like(b) for b in biases]

batch_size = 64
epochs = 45
transform_stop_epoch = 30
learning_rate = 0.05
beta = 0.9

In [731]:
for j in range(epochs):
    indices = np.random.permutation(len(images_scaled))
    X_shuffled = images_scaled[indices]
    Y_shuffled = labels[indices]

    use_transform = j < transform_stop_epoch

    for i in range(len(images_scaled) // batch_size):
        batch_X = X_shuffled[i * batch_size : (i + 1) * batch_size]
        batch_Y = Y_shuffled[i * batch_size : (i + 1) * batch_size]

        if use_transform:
            random_angle = np.random.uniform(-15, 15)
            sx = np.random.uniform(-2, 2)
            sy = np.random.uniform(-2, 2)
            transformed_batch_X = transform_batch(batch_X, random_angle, sx, sy)
        else:
            transformed_batch_X = batch_X

        w_updates, b_updates = neural_step_batch_update_any(
            transformed_batch_X,
            batch_Y,
            weights,
            biases,
            learning_rate,
        )

        for k in range(len(weights)):
            w_vel[k] = (beta * w_vel[k]) + w_updates[k]
            b_vel[k] = (beta * b_vel[k]) + b_updates[k]

            weights[k] -= w_vel[k]
            biases[k] -= b_vel[k]

    learning_rate *= 0.96

In [732]:
a_test = test_images_scaled.T

for i in range(len(weights) - 1):
    z_test = weights[i] @ a_test + biases[i]
    a_test = ReLU(z_test)

z_final = weights[-1] @ a_test + biases[-1]
output = np.argmax(z_final, axis=0)

accuracy = np.mean(output == test_labels) * 100
print(f"Final Test Accuracy: {accuracy:.2f}%")

Final Test Accuracy: 99.08%


HOOOLY WE DID IT

Only thing left is pytortch and Cnn's

QR least sqaures for the last step?